In [ ]:
# %% [Task 0 — Setup and template]
# Principle 5: templates handle style globally — define once, every chart inherits it.

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Shared palette (same as the Zambia notebook for course continuity)
PALETTE = ['#7BB3B2','#65A6BD','#C997AF','#B8B0D3','#F4CF97','#98B9A0','#F6DECD']

# Build the template once
nso_template = go.layout.Template()
nso_template.layout = go.Layout(
    font=dict(family='Arial', size=13, color='#333'),
    title_font=dict(size=16, color='#222'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    colorway=PALETTE,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=60, r=30, t=60, b=40),
)
pio.templates['nso'] = nso_template
pio.templates.default = 'nso'

# Plotly toolbar config (clean modebar, sensible PNG export)
PLOTLY_CONFIG = {
    'displaylogo': False,
    'modeBarButtonsToRemove': ['select2d', 'lasso2d', 'autoScale2d'],
    'toImageButtonOptions': {'format': 'png', 'width': 1200, 'height': 700, 'scale': 2},
}

# Load the analytical parquet from the previous exercise
df = pd.read_parquet('data/02_processed/ces_tanzania_2025_analytic.parquet')

print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes.value_counts())
df.head(3)

In [ ]:
# %% [Task 1 — Firm-age distribution with median reference]
# Principle 1: one trace + layout. Principle 4: the reference line is layout chrome.

# ----- Step 1: Prepare -----
age = df['firm_age'].dropna()
median_age = age.median()

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Histogram(x=age, nbinsx=20, marker_color=PALETTE[0], name='Firm age')
)

fig.add_vline(
    x=median_age, line_dash='dash', line_color='red',
    annotation_text=f'Median: {median_age:.0f} years',
    annotation_position='top right',
)

fig.update_layout(
    title='Distribution of establishment age — Tanzania CES 2025',
    xaxis_title='Establishment age (years)',
    yaxis_title='Number of firms',
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 2 — Number of establishments by broad sector]
# Principle 2: one trace per visual series. Principle 5: prepare first, plot second.

# ----- Step 1: Prepare -----
sector = df['sector_name'].value_counts().sort_values()

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Bar(
        y=sector.index, x=sector.values, orientation='h',
        marker_color=PALETTE[0],
        text=sector.values, textposition='outside',
    )
)

fig.update_layout(
    title='Number of establishments by broad sector',
    xaxis_title='Number of firms',
    height=max(400, len(sector) * 30),
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 3 — Establishments interviewed by region]
# Principle 2 again. With ~25–30 categories we rotate x labels rather than reshape the data.

# ----- Step 1: Prepare -----
region = df['region_name'].value_counts().sort_values(ascending=False)

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Bar(x=region.index, y=region.values, marker_color=PALETTE[1])
)

fig.update_layout(
    title='Establishments interviewed by region',
    yaxis_title='Number of firms',
    xaxis_tickangle=-45,
    height=500,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 4 — Establishment size distribution (donut)]
# Principle 1: pie is still just one trace + layout.
# Use pies only with 2–4 categories — size_band has exactly 4.

# ----- Step 1: Prepare -----
size_counts = df['size_band'].value_counts()

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Pie(
        labels=size_counts.index,
        values=size_counts.values,
        hole=0.4,
        marker=dict(colors=PALETTE[:4], line=dict(color='white', width=1.5)),
        textinfo='percent+label',
    )
)

fig.update_layout(title='Establishment size distribution')
fig.show(config=PLOTLY_CONFIG)

# Note: we deliberately do NOT pie-chart legal_status_name (6 categories — too many for a pie).
# It appears as a horizontal bar in the 2x2 subplots below instead.

In [ ]:
# %% [Task 5 — Firm landscape, 2x2 subplots]
# Principle 3: multiple things in one figure = multiple traces, via make_subplots.

# ----- Step 1: Prepare -----
legal = df['legal_status_name'].value_counts().sort_values()
sector_top = df['sector_name'].value_counts().head(10).sort_values()
region_top = df['region_name'].value_counts().head(10).sort_values()
exporter = (
    df['is_exporter']
    .map({True: 'Exporter', False: 'Non-exporter'})
    .value_counts()
)

# ----- Step 2: Plot -----
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Legal status', 'Top 10 sectors',
                    'Top 10 regions', 'Exporters vs non-exporters'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'pie'}]],
)

fig.add_trace(
    go.Bar(y=legal.index, x=legal.values, orientation='h',
           marker_color=PALETTE[0], showlegend=False),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(y=sector_top.index, x=sector_top.values, orientation='h',
           marker_color=PALETTE[1], showlegend=False),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(y=region_top.index, x=region_top.values, orientation='h',
           marker_color=PALETTE[2], showlegend=False),
    row=2, col=1,
)
fig.add_trace(
    go.Pie(labels=exporter.index, values=exporter.values,
           marker=dict(colors=PALETTE[:len(exporter)]),
           textinfo='percent+label', showlegend=False),
    row=2, col=2,
)

fig.update_layout(title='Tanzanian establishments at a glance', height=800)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 6 — Sales distribution + box-by-size subplots]
# Principle 3: mixed chart types in one figure.
# Principle 4: log scale is a layout property — we transform manually for an interpretable axis.

# ----- Step 1: Prepare -----
sales = df['s7q26'].dropna()
sales = sales[sales > 0]
log_sales = np.log10(sales)
median_sales = sales.median()

size_order = ['Micro', 'Small', 'Medium', 'Large']

# ----- Step 2: Plot -----
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribution (log10 TZS)', 'By size band'),
)

fig.add_trace(
    go.Histogram(x=log_sales, nbinsx=30, marker_color=PALETTE[0],
                 name='Sales', showlegend=False),
    row=1, col=1,
)

for i, band in enumerate(size_order):
    subset = df[df['size_band'] == band]['s7q26'].dropna()
    subset = subset[subset > 0]
    fig.add_trace(
        go.Box(y=np.log10(subset), name=band, marker_color=PALETTE[i]),
        row=1, col=2,
    )

fig.add_vline(
    x=np.log10(median_sales), line_dash='dash', line_color='red',
    annotation_text=f'Median: {median_sales:,.0f} TZS',
    row=1, col=1,
)

fig.update_layout(title='Total annual sales (TZS, log scale) — distribution and by firm size')
fig.update_xaxes(title_text='log10(sales, TZS)', row=1, col=1)
fig.update_yaxes(title_text='Number of firms', row=1, col=1)
fig.update_yaxes(title_text='log10(sales, TZS)', row=1, col=2)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 7 — Ownership composition by sector, stacked horizontal bar]
# Principle 4: barmode='stack' lives in the layout.
# Principle 5: the four ownership columns already exist in the parquet — plotting is trivial.

# ----- Step 1: Prepare -----
own_cols = ['own_private_domestic', 'own_private_foreign', 'own_government', 'own_other']
own_labels = ['Private domestic', 'Private foreign', 'Government', 'Other']

top_sectors = df['sector_name'].value_counts().head(6).index
own_by_sector = (
    df[df['sector_name'].isin(top_sectors)]
    .groupby('sector_name')[own_cols]
    .mean()
    .loc[top_sectors]  # preserve top-6 ordering
)

# ----- Step 2: Plot -----
fig = go.Figure()
for col, label, color in zip(own_cols, own_labels, PALETTE[:4]):
    fig.add_trace(
        go.Bar(
            y=own_by_sector.index, x=own_by_sector[col],
            name=label, orientation='h', marker_color=color,
        )
    )

fig.update_layout(
    title='Average ownership composition by sector (%)',
    xaxis_title='Share of ownership (%)',
    barmode='stack',
    height=450,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 8 — Obstacles heatmap]
# Principle 1: one Heatmap trace + layout.
# Principle 5: all the work is in the groupby; the chart is a one-liner.

# ----- Step 1: Prepare -----
obstacle_cols = ['s9q11', 's12q10', 's15q8', 's16q33', 's3q33a', 's3q33b']
obstacle_labels = ['Electricity', 'Informal competition', 'Access to land',
                   'Access to finance', 'Labour regulations', 'Workforce education']

heatmap_data = (
    df.groupby('sector_name')[obstacle_cols]
    .mean()
    .rename(columns=dict(zip(obstacle_cols, obstacle_labels)))
)

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns.tolist(),
        y=heatmap_data.index.tolist(),
        text=heatmap_data.round(1).values,
        texttemplate='%{text:.1f}',
        colorscale='YlOrRd',
        colorbar=dict(title='Mean severity<br>(1=none, 5=very severe)'),
    )
)

fig.update_layout(
    title='Average severity of business obstacles by sector',
    height=450,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 9 — Diverging Likert bar: access to finance as obstacle]
# Principle 2: one trace per Likert level.
# Principle 4: barmode='relative' is a layout setting that does the diverging magic.

# ----- Step 1: Prepare -----
fin_levels = [1, 2, 3, 4, 5]
fin_labels = ['No obstacle', 'Minor', 'Moderate', 'Major', 'Very severe']

pct = (
    df['s16q33'].dropna()
    .value_counts(normalize=True)
    .reindex(fin_levels, fill_value=0)
    .mul(100)
)

# Red-to-blue diverging palette: 1-2 cool, 3 neutral, 4-5 hot
diverging_colors = ['#4575b4', '#91bfdb', '#fee090', '#fc8d59', '#d73027']

# ----- Step 2: Plot -----
fig = go.Figure()
for i, (level, label, value, color) in enumerate(zip(fin_levels, fin_labels, pct, diverging_colors)):
    # Levels 1–2 pull left (negative x), levels 3–5 pull right (positive x)
    x_val = -value if i < 2 else value
    fig.add_trace(
        go.Bar(
            y=[' '], x=[x_val], name=label, orientation='h',
            marker_color=color,
            text=[f'{value:.0f}%' if value >= 4 else ''],
            textposition='inside', textfont=dict(color='white'),
        )
    )

fig.add_vline(x=0, line_color='gray', line_width=1)
fig.update_layout(
    title='How severe an obstacle is access to finance? (% of firms)',
    xaxis_title='Share of respondents',
    barmode='relative',
    height=250,
)
fig.show(config=PLOTLY_CONFIG)

# Homework: replicate this chart for s9q11 (electricity) and s15q8 (access to land).

In [ ]:
# %% [Task 10 — Exporter vs non-exporter by sector, grouped bar]
# Principle 2: one trace per group.

# ----- Step 1: Prepare -----
top_sectors = df['sector_name'].value_counts().head(8).index
tmp = df[df['sector_name'].isin(top_sectors)].copy()

ct = (
    pd.crosstab(tmp['sector_name'], tmp['is_exporter'], normalize='index')
    .mul(100)
    .reindex(top_sectors)
)
ct.columns = ['Non-exporter' if c is False else 'Exporter' for c in ct.columns]

# ----- Step 2: Plot -----
fig = go.Figure()
for i, group in enumerate(['Exporter', 'Non-exporter']):
    fig.add_trace(
        go.Bar(x=ct.index, y=ct[group], name=group, marker_color=PALETTE[i])
    )

fig.update_layout(
    title='Share of exporters vs non-exporters by sector',
    yaxis_title='Share of firms (%)',
    barmode='group',
    xaxis_tickangle=-30,
    height=500,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 11 — Adoption of business practices, stacked Yes/No]
# Principle 2 + Principle 5: same recipe as the Zambia savings chart.

# ----- Step 1: Prepare -----
practice_cols = {
    's13q1': 'Introduced new/improved product',
    's3q10': 'Provided formal training',
    's10q6': 'Monitors performance indicators',
    's17q1': 'Uses mobile money',
}

rows = []
for col, label in practice_cols.items():
    s = df[col].map({1: 'Yes', 2: 'No'}).dropna()
    counts = s.value_counts()
    total = counts.sum()
    rows.append({
        'practice': label,
        'Yes': counts.get('Yes', 0) / total * 100,
        'No': counts.get('No', 0) / total * 100,
    })

plot_df = pd.DataFrame(rows).set_index('practice').sort_values('Yes')

# ----- Step 2: Plot -----
fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=plot_df.index, x=plot_df['Yes'], name='Yes', orientation='h',
        marker_color=PALETTE[0],
        text=[f'{v:.0f}%' for v in plot_df['Yes']],
        textposition='inside', textfont=dict(color='white'),
    )
)
fig.add_trace(
    go.Bar(
        y=plot_df.index, x=plot_df['No'], name='No', orientation='h',
        marker_color='#D3D3D3',
    )
)

fig.update_layout(
    title='Adoption of key business practices among Tanzanian establishments',
    xaxis_title='Share of firms (%)',
    xaxis_range=[0, 100],
    barmode='stack',
    height=400,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 12 — Firm size vs total annual sales, scatter with log axes]
# Principle 1: one Scatter trace per group. Principle 4: yaxis_type='log' is layout.

# ----- Step 1: Prepare -----
plot_df = df[['Workers', 's7q26', 'is_exporter']].copy()
plot_df = plot_df[(plot_df['Workers'] > 0) & (plot_df['s7q26'] > 0)].dropna()

# ----- Step 2: Plot -----
# Splitting into two traces (one per exporter status) so the legend distinguishes them.
fig = go.Figure()
for i, (flag, label) in enumerate([(True, 'Exporter'), (False, 'Non-exporter')]):
    subset = plot_df[plot_df['is_exporter'] == flag]
    fig.add_trace(
        go.Scatter(
            x=subset['Workers'], y=subset['s7q26'],
            mode='markers', name=label,
            marker=dict(color=PALETTE[i], size=6, opacity=0.45),
        )
    )

fig.update_layout(
    title='Firm size vs total annual sales (log scale)',
    xaxis_title='Total workers',
    yaxis_title='Total annual sales (TZS)',
    xaxis_type='log',
    yaxis_type='log',
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Capstone — 1x2 subplot combining Tasks 8 and 9]
# Nothing new conceptually: make_subplots + add_trace(..., row=, col=).

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.65, 0.35],
    subplot_titles=('Obstacle severity by sector', 'Access to finance — % of firms'),
    specs=[[{'type': 'heatmap'}, {'type': 'bar'}]],
)

# Left: heatmap (reuse heatmap_data from Task 8)
fig.add_trace(
    go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns.tolist(),
        y=heatmap_data.index.tolist(),
        colorscale='YlOrRd',
        colorbar=dict(title='Severity', x=0.6),
        showscale=True,
    ),
    row=1, col=1,
)

# Right: diverging Likert (reuse pct from Task 9)
for i, (label, value, color) in enumerate(zip(fin_labels, pct, diverging_colors)):
    x_val = -value if i < 2 else value
    fig.add_trace(
        go.Bar(
            y=[' '], x=[x_val], name=label, orientation='h',
            marker_color=color, showlegend=True,
        ),
        row=1, col=2,
    )

fig.update_layout(
    title='The business environment in Tanzania, 2025',
    barmode='relative',
    height=500,
)
fig.show(config=PLOTLY_CONFIG)